**Demo: Writing Invoice data to partitions**

In [0]:
spark.sql("create database if not exists stage_wwi")
spark.sql("use stage_wwi")
spark.sql("drop table if exists invoicep")
spark.sql("drop table if exists invoicepy")
spark.sql("drop table if exists invoicepyq")


In [0]:
# File location and type
file_location = "/Volumes/workspace/stage_wwi/raw_data/Invoice.csv"
file_type = "csv"

# CSV options
infer_schema = "true"
first_row_is_header = "true"
delimiter = ","

# The applied options are for CSV files. For other file types, these will be ignored.
dfinvoice = spark.read.format(file_type) \
  .option("inferSchema", infer_schema) \
  .option("header", first_row_is_header) \
  .option("sep", delimiter) \
  .load(file_location)

# display(dfinvoice)

In [0]:
# df2021 = dfinvoice.filter((dfinvoice.InvoiceDate >= "2021-01-01") & (dfinvoice.InvoiceDate <= "2021-12-31"))

dfinvoice.write.format("delta").mode("overwrite").saveAsTable("invoicep")

In [0]:
%sql

SELECT  YEAR(InvoiceDate), MONTH(InvoiceDate), COUNT(1) AS RowCount
  from invoicep
  WHERE InvoiceDate >= '2022-01-01' AND InvoiceDate <= '2022-12-31'
  GROUP BY YEAR(InvoiceDate), MONTH(InvoiceDate)
  ORDER BY YEAR(InvoiceDate), MONTH(InvoiceDate)
  

In [0]:
%sql
DESCRIBE HISTORY workspace.stage_wwi.invoicepy

In [0]:

dfinvoicepy = spark.sql("""SELECT InvoiceID, InvoiceDate, YEAR(InvoiceDate) AS InvoiceYear
    , CustomerID, BillToCustomerID, OrderID, DeliveryMethodID, ContactPersonID
    , AccountsPersonID, SalespersonPersonID, PackedByPersonID, CustomerPurchaseOrderNumber, IsCreditNote
    , CreditNoteReason, Comments, DeliveryInstructions, InternalComments
    , LastEditedBy, LastEditedWhen
   FROM invoicep 
   WHERE InvoiceDate <= '2024-05-01'""")
# display(dfinvoicepy)
  
dfinvoicepy.write.format("delta").partitionBy("InvoiceYear").mode("append").saveAsTable("invoicepy")

In [0]:
%sql
DESCRIBE HISTORY workspace.stage_wwi.invoicepy

In [0]:
%sql

SELECT  YEAR(InvoiceDate), MONTH(InvoiceDate), COUNT(1) AS RowCount
  from invoicepy
  WHERE InvoiceYear = 2022
  GROUP BY YEAR(InvoiceDate), MONTH(InvoiceDate)
  ORDER BY YEAR(InvoiceDate), MONTH(InvoiceDate)

In [0]:
dfinvoicepy1d = spark.sql("""SELECT InvoiceID, InvoiceDate, YEAR(InvoiceDate) AS InvoiceYear
    , CustomerID, BillToCustomerID, OrderID, DeliveryMethodID, ContactPersonID
    , AccountsPersonID, SalespersonPersonID, PackedByPersonID, CustomerPurchaseOrderNumber, IsCreditNote
    , CreditNoteReason, Comments, DeliveryInstructions, InternalComments
    , LastEditedBy, LastEditedWhen
   FROM invoicep 
   WHERE InvoiceDate = '2024-05-01'""")
# display(dfinvoicepy)
dfinvoicepy1d.write.format("delta").partitionBy("InvoiceYear").mode("append").saveAsTable("invoicepy")

In [0]:
dfinvoicepy2d = spark.sql("""SELECT InvoiceID, InvoiceDate, YEAR(InvoiceDate) AS InvoiceYear
    , CustomerID, BillToCustomerID, OrderID, DeliveryMethodID, ContactPersonID
    , AccountsPersonID, SalespersonPersonID, PackedByPersonID, CustomerPurchaseOrderNumber, IsCreditNote
    , CreditNoteReason, Comments, DeliveryInstructions, InternalComments
    , LastEditedBy, LastEditedWhen
   FROM invoicep 
   WHERE InvoiceDate = '2024-05-02'""")
# display(dfinvoicepy)
dfinvoicepy2d.write.format("delta").partitionBy("InvoiceYear").mode("append").saveAsTable("invoicepy")

In [0]:
dfinvoicepy3d = spark.sql("""SELECT InvoiceID, InvoiceDate, YEAR(InvoiceDate) AS InvoiceYear
    , CustomerID, BillToCustomerID, OrderID, DeliveryMethodID, ContactPersonID
    , AccountsPersonID, SalespersonPersonID, PackedByPersonID, CustomerPurchaseOrderNumber, IsCreditNote
    , CreditNoteReason, Comments, DeliveryInstructions, InternalComments
    , LastEditedBy, LastEditedWhen
   FROM invoicep 
   WHERE InvoiceDate = '2024-05-03'""")
# display(dfinvoicepy)
dfinvoicepy3d.write.format("delta").partitionBy("InvoiceYear").mode("append").saveAsTable("invoicepy")

In [0]:
%sql

SELECT  YEAR(InvoiceDate), MONTH(InvoiceDate), COUNT(1) AS RowCount
  from invoicepy
  WHERE InvoiceDate >= '2024-04-01'
  GROUP BY YEAR(InvoiceDate), MONTH(InvoiceDate)
  ORDER BY YEAR(InvoiceDate), MONTH(InvoiceDate)
  

**Partition by Quarter**

In [0]:
spark.sql("drop table if exists invoicepyq")
dfinvoicepyq = spark.sql("""SELECT  
    i.InvoiceID, i.InvoiceDate
    , YEAR(i.InvoiceDate)  AS InvoiceYear
    , CONCAT(CAST(YEAR(i.InvoiceDate) AS String), "Q", CAST(QUARTER(i.InvoiceDate) AS STRING)) AS InvoiceYearQtr
    , i.CustomerID, i.BillToCustomerID
  FROM invoice i
  """)
dfinvoicepyq.display()
dfinvoicepyq.write.format("delta").partitionBy("InvoiceYearQtr").mode("append").saveAsTable("invoicepyq")

In [0]:
%sql
DESCRIBE HISTORY workspace.stage_wwi.invoicepyq